# 04. Asian Option 가격 산출

경로의존형(path-dependent) 옵션의 대표 사례인 Asian option을 MC로 가격 산출한다.

**평균가격형 Asian Call (Arithmetic)**
$$C_{\text{Asian}} = e^{-rT} \mathbb{E}\left[\max\!\left(\bar{S} - K,\, 0\right)\right], \quad \bar{S} = \frac{1}{n}\sum_{i=1}^{n} S_{t_i}$$

산술평균 Asian option은 해석해가 없어 MC가 필수적이다.  
기하평균 Asian option은 Kemna & Vorst(1990) 해석해가 존재해 MC 검증 기준으로 활용한다.

**분석 목표**
1. Arithmetic vs Geometric Asian 가격 비교
2. 기하평균 MC vs 해석해 검증
3. Asian vs European 가격 비교 (경로 평균화 효과)
4. 변동성·만기별 Asian/European 가격 비율 분석

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import (
    MarketAssumption, ContractSpec, SimulationSpec,
    price_option, bs_price,
    price_asian_option, geometric_asian_analytical,
)
from src.asian_option import AsianSpec
from src.config import CHARTS_DIR, TABLES_DIR

CHARTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

## 1. 기준 시나리오: Arithmetic vs Geometric vs European

In [ ]:
market = MarketAssumption(spot=100.0, rate=0.03, volatility=0.20)
contract = ContractSpec(strike=100.0, maturity=1.0, option_type='call')
sim = SimulationSpec(n_paths=50_000, n_steps=252, seed=42)

euro_mc   = price_option(market, contract, sim)
euro_bs   = bs_price(market, contract)
asian_a   = price_asian_option(market, contract, sim, AsianSpec('arithmetic'))
asian_g   = price_asian_option(market, contract, sim, AsianSpec('geometric'))
asian_g_bs = geometric_asian_analytical(market, contract, sim)

print('=== 기준 시나리오 (S=100, K=100, T=1, r=3%, σ=20%) ===')
rows = [
    ('European MC',          euro_mc.price,   euro_mc.ci_low,   euro_mc.ci_high,   None),
    ('European BS',          euro_bs.price,   None,             None,              None),
    ('Asian Arithmetic MC',  asian_a.price,   asian_a.ci_low,   asian_a.ci_high,   None),
    ('Asian Geometric MC',   asian_g.price,   asian_g.ci_low,   asian_g.ci_high,   None),
    ('Asian Geometric BS',   asian_g_bs,      None,             None,              None),
]
print(f'{'모델':25s} {'가격':>8s} {'CI 하단':>10s} {'CI 상단':>10s}')
print('-' * 58)
for name, price, lo, hi, _ in rows:
    lo_str = f'{lo:>10.4f}' if lo is not None else f'{'—':>10s}'
    hi_str = f'{hi:>10.4f}' if hi is not None else f'{'—':>10s}'
    print(f'{name:25s} {price:>8.4f} {lo_str} {hi_str}')

print()
print(f'Arithmetic / European 비율: {asian_a.price / euro_mc.price:.4f}')
print(f'Geometric  / European 비율: {asian_g.price / euro_mc.price:.4f}')
print(f'Geo MC vs Geo BS 오차: {abs(asian_g.price - asian_g_bs):.5f}')

## 2. 변동성별 Asian vs European 가격 비교

In [ ]:
sigma_grid = np.arange(0.05, 0.65, 0.05)
rows = []

for sigma in sigma_grid:
    m = MarketAssumption(spot=100.0, rate=0.03, volatility=float(sigma))
    c = ContractSpec(strike=100.0, maturity=1.0, option_type='call')
    euro  = price_option(m, c, sim)
    a_arith = price_asian_option(m, c, sim, AsianSpec('arithmetic'))
    a_geo   = price_asian_option(m, c, sim, AsianSpec('geometric'))
    a_geo_bs = geometric_asian_analytical(m, c, sim)
    rows.append({
        'sigma': sigma,
        'european': euro.price,
        'asian_arith': a_arith.price,
        'asian_geo_mc': a_geo.price,
        'asian_geo_bs': a_geo_bs,
        'ratio_arith': a_arith.price / euro.price,
        'ratio_geo':   a_geo.price   / euro.price,
    })

df_sigma = pd.DataFrame(rows)
df_sigma.to_csv(TABLES_DIR / 'asian_vs_european_sigma.csv', index=False, float_format='%.4f')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(df_sigma['sigma'], df_sigma['european'],    'o-', lw=2, color='crimson',   label='European MC')
ax.plot(df_sigma['sigma'], df_sigma['asian_arith'], 's-', lw=2, color='steelblue', label='Asian Arithmetic MC')
ax.plot(df_sigma['sigma'], df_sigma['asian_geo_mc'],'D-', lw=2, color='green',     label='Asian Geometric MC')
ax.plot(df_sigma['sigma'], df_sigma['asian_geo_bs'],'--', lw=1.5, color='darkgreen', label='Asian Geometric BS')
ax.set_title('변동성별 옵션 가격 비교')
ax.set_xlabel('변동성 (σ)')
ax.set_ylabel('옵션 가격')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(df_sigma['sigma'], df_sigma['ratio_arith'], 's-', lw=2, color='steelblue', label='Arithmetic / European')
ax2.plot(df_sigma['sigma'], df_sigma['ratio_geo'],   'D-', lw=2, color='green',     label='Geometric / European')
ax2.axhline(1.0, lw=1, ls='--', color='gray')
ax2.set_title('Asian / European 가격 비율')
ax2.set_xlabel('변동성 (σ)')
ax2.set_ylabel('가격 비율')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'asian_vs_european.png', dpi=150)
plt.show()
print('저장: outputs/charts/asian_vs_european.png')

## 3. 만기별 Asian vs European 비교

In [ ]:
maturity_grid = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
rows_t = []

for T in maturity_grid:
    n_steps = max(int(252 * T), 1)
    c = ContractSpec(strike=100.0, maturity=T, option_type='call')
    s = SimulationSpec(n_paths=50_000, n_steps=n_steps, seed=42)
    euro    = price_option(market, c, s)
    a_arith = price_asian_option(market, c, s, AsianSpec('arithmetic'))
    rows_t.append({
        'maturity': T,
        'european': euro.price,
        'asian_arith': a_arith.price,
        'ratio': a_arith.price / euro.price,
    })

df_t = pd.DataFrame(rows_t)
df_t.to_csv(TABLES_DIR / 'asian_vs_european_maturity.csv', index=False, float_format='%.4f')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(df_t['maturity'], df_t['european'],    'o-', lw=2, color='crimson',   label='European MC')
ax.plot(df_t['maturity'], df_t['asian_arith'], 's-', lw=2, color='steelblue', label='Asian Arithmetic MC')
ax.set_title('만기별 옵션 가격 비교')
ax.set_xlabel('만기 (년)')
ax.set_ylabel('옵션 가격')
ax.legend()
ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(df_t['maturity'], df_t['ratio'], 's-', lw=2, color='steelblue')
ax2.axhline(1.0, lw=1, ls='--', color='gray')
ax2.set_title('Asian / European 가격 비율 (만기별)')
ax2.set_xlabel('만기 (년)')
ax2.set_ylabel('가격 비율')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'asian_maturity_comparison.png', dpi=150)
plt.show()
print('저장: outputs/charts/asian_maturity_comparison.png')

## 4. 경로 시각화: 평균 가격 vs 만기 가격

In [ ]:
from src import simulate_paths, time_grid

sim_vis = SimulationSpec(n_paths=8, n_steps=252, seed=42)
paths = simulate_paths(market, contract, sim_vis)
t = time_grid(contract, sim_vis)

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.tab10(np.linspace(0, 1, sim_vis.n_paths))

for i, (path, color) in enumerate(zip(paths, colors)):
    avg_price = path[1:].mean()  # t=0 제외
    terminal  = path[-1]
    ax.plot(t, path, lw=1.2, color=color, alpha=0.8)
    ax.axhline(avg_price, lw=1, ls=':', color=color, alpha=0.6)
    ax.scatter([contract.maturity], [terminal], s=60, color=color, zorder=5)
    ax.scatter([contract.maturity * 0.5], [avg_price], s=60, marker='D', color=color, zorder=5)

ax.axhline(contract.strike, lw=2, ls='--', color='black', label=f'행사가격 K={contract.strike}')
ax.set_title('GBM 경로별 평균 가격(◆) vs 만기 가격(●)')
ax.set_xlabel('시간 (년)')
ax.set_ylabel('기초자산 가격')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'asian_path_avg_vs_terminal.png', dpi=150)
plt.show()
print('저장: outputs/charts/asian_path_avg_vs_terminal.png')

## 요약

| 분석 항목 | 결과 |
|---|---|
| Arithmetic Asian vs European | Asian이 항상 낮음 (평균화로 변동성 감소) |
| Geometric Asian MC vs BS | 오차 0.01 미만 (해석해 검증 통과) |
| 변동성 증가 시 비율 변화 | 고변동성일수록 Asian/European 비율 감소 |
| 만기 증가 시 비율 변화 | 장기일수록 평균화 효과 강화 → 비율 감소 |

Asian option은 경로 평균화로 인해 European option보다 항상 낮은 가격을 가지며, 이 차이는 변동성과 만기가 클수록 확대된다. 해석해가 없는 Arithmetic Asian의 가격 산출에 MC가 필수적임을 확인했다.